# OPEN5GS throughput probe trial

Three Gent VMs: node1 hosts the core, Kubernetes control plane, monitoring and iperf server; node2 hosts one monolithic OAI RFsim gNB; node3 hosts the UE(s). Two UEs each use their own slice and SMF/UPF pair.

Uses stock upstream core images. Checks session mapping and both latency and throughput probes with uplink/downlink TCP traffic. No RAN KPI calculator, metrics-parser customization, Loki or packet pairing. The trial runner destroys this deployment after validation, including on failure.

In [ ]:
from pathlib import Path
from datetime import UTC, datetime
import asyncio
import json
import os
import sys

REPOSITORY = Path("../../..").resolve()


In [ ]:
# Refresh upstream and use this local monitoring checkout.
%pip install --upgrade --force-reinstall --no-deps "orchestrator @ git+https://gitlab.inria.fr/slices-ri/blueprints/post-5g/orchestrator.git@main"
%pip install -e "{REPOSITORY}" --extra-index-url=https://gitlab.inria.fr/api/v4/projects/70914/packages/pypi/simple
os.environ["PATH"] = str(Path(sys.executable).parent) + os.pathsep + os.environ["PATH"]
sys.path.insert(0, str(REPOSITORY / "src"))


In [ ]:
!{sys.executable} -m ansible.cli.galaxy install -r "{REPOSITORY / 'ansible' / 'requirements.yaml'}"


## Topology

In [ ]:
CORE_IMPLEMENTATION = 'open5gs'
PROJECT = "post5g-beta"
INFRA = "be-gent1-bi-vm1"
EXPERIMENT = "throughput-open5gs-" + datetime.now(UTC).strftime("%Y%m%dT%H%M%S%fZ")

SLICE_CONFIG = {
    "slice1": {
        "sst": 1,
        "sd": None,
        "dnn": "internet",
        "network": "12.1.1.0/24",
        "smf": "smf1",
        "upf": "upf1"
    },
    "slice2": {
        "sst": 1,
        "sd": "100000",
        "dnn": "streaming",
        "network": "14.1.1.0/24",
        "smf": "smf2",
        "upf": "upf2"
    }
}
CLIENTS = [f"ue{i}" for i in range(1, len(SLICE_CONFIG) + 1)]
UE_SLICES = dict(zip(CLIENTS, SLICE_CONFIG))
UE_GROUPS = {"gnb": CLIENTS}
print(f"Experiment: {EXPERIMENT}; 3 VMs; {len(CLIENTS)} UE(s)")


In [ ]:
from metamodel.models.metamodel import MetaModel

core_components = {
    name: component.model_dump(mode="json")
    for name, component in MetaModel().core.components.items()
    if component.type not in ("smf", "upf")
    and (CORE_IMPLEMENTATION == "open5gs" or name in {"amf", "ausf", "nrf", "pcf", "udm", "udr"})
}
for slice_name, config in SLICE_CONFIG.items():
    core_components[config["smf"]] = {
        "type": "smf", "config": {"snssaiList": [slice_name], "upfs": [config["upf"]]},
    }
    core_components[config["upf"]] = {"type": "upf", "config": {"snssaiList": [slice_name]}}
for name in ("amf", "nrf"):
    core_components[name]["config"]["plmnList"] = [{"snssaiList": list(SLICE_CONFIG)}]
if "nssf" in core_components:
    core_components["nssf"]["config"] = {
        "nsiInfoList": [
            {"snssai": name, "nsiInformationList": [{"nrfId": "http://open5gs-nrf:80", "nsiId": i}]}
            for i, name in enumerate(SLICE_CONFIG, start=1)
        ],
        "taInfoList": [{"snssaiList": list(SLICE_CONFIG)}],
        "amfInfoList": [],
    }
metamodel = MetaModel.model_validate({
    "core": {"components": core_components},
    "slices": {
        name: {"sst": config["sst"], "sd": config["sd"],
               "dnns": {config["dnn"]: {"ipv4Network": config["network"]}}}
        for name, config in SLICE_CONFIG.items()
    },
    "ran": {"components": {"gnb": {"type": "gnb", "config": {"plmnList": [
        {"snssaiList": list(SLICE_CONFIG)}], "ru": "sim_ru"}}}},
    "rus": {"sim_ru": {"type": "simulated"}},
    "ues": {
        name: {"config": {
            "supi": f"00101{100 + i:010d}", "gnbs": ["gnb"], "ru": "sim_ru",
            "subscriptions": [{"snssai": UE_SLICES[name], "dnn": SLICE_CONFIG[UE_SLICES[name]]["dnn"]}],
        }}
        for i, name in enumerate(CLIENTS)
    },
})


In [ ]:
from orchestrator.models.deployment import DeploymentModel

deployment_model = DeploymentModel.model_validate({
    "params": {
        "provisioning": {
            "project": PROJECT, "experiment": EXPERIMENT, "infra": INFRA,
            "flavor": "large", "extra_path": "~/slices/bin", "create_experiment": True,
        },
        "k8s": {
            "master_node": "node1",
            "local_kubeconfig_path": str(Path(f".k8s-{EXPERIMENT}.conf").resolve()),
        },
    },
    "nodes": {
        "node1": {"roles": ["core", "database"]},
        "node2": {"roles": ["ran"]},
        "node3": {"roles": ["ue"]},
    },
    "core": {"implementation": CORE_IMPLEMENTATION},
    "ran": {"implementation": "oai", "components": {"gnb": {"type": "gnb", "node_name": "node2"}}},
    "ues": {
        name: {"type": "containerized", "implementation": "oai", "node_name": "node3"}
        for name in CLIENTS
    },
})


## Provision and deploy Kubernetes

In [ ]:
from orchestrator.deployment import ProvisioningManagement, K8SManagement

prov = ProvisioningManagement(metamodel=metamodel, deployment_model=deployment_model)
await prov.provision_nodes(nodes=["node1"])
await prov.provision()
print(prov.get_ansible_inventory())


In [ ]:
k8s = K8SManagement(preceding=prov)
await k8s.deploy()


## Deploy the stock core

In [ ]:
from orchestrator.deployment import CoreManagement

core = CoreManagement(preceding=k8s)
for component in core.deployment_model.core.components.values():
    component.node_name = "node1"


In [ ]:
# Keep the upstream stock images and templates; retain the scenario's SMF slice advertisement.
from orchestrator.models.core import ADVANCED_CORE_CONFIG_MAPPING

for name, component in core.deployment_model.core.components.items():
    if component.type != "smf":
        continue
    advanced = component.advanced_config.model_dump() if component.advanced_config else {}
    advanced["info"] = {"s_nssai": []}
    for slice_name in core.metamodel.core.components[name].config.snssaiList:
        snssai = core.metamodel.slices[slice_name]
        entry = {"sst": snssai.sst, "dnn": list(snssai.dnns)}
        if snssai.sd is not None:
            entry["sd"] = f"{snssai.sd:06x}"
        advanced["info"]["s_nssai"].append(entry)
    config_type = ADVANCED_CORE_CONFIG_MAPPING[component.implementation][component.type]
    component.advanced_config = config_type.model_validate(advanced)


In [ ]:
await core.run_database_action(action="prepare")
await core.prepare()
await core.run_database_action(action="install")
await core.install()
await core.register_ues()


## Deploy one RFsim gNB, then inject monitoring before UE attachment

In [ ]:
from orchestrator.deployment import RANManagement

ran = RANManagement(preceding=core)
await ran.deploy()


In [ ]:
from orchestrator_monitoring import MonitoringManagement
from orchestrator_monitoring.models import MonitoringDeployment

QUERIES_FILE = Path("queries.json")
monitoring = MonitoringManagement(
    preceding=ran,
    config=MonitoringDeployment.from_yaml_file("monitoring.yaml"),
)
await monitoring.deploy()


## Attach the UE(s) and prepare short uplink/downlink transfers

In [ ]:
from orchestrator.deployment import UEManagement

ue = UEManagement(preceding=ran)
for client in CLIENTS:
    print(f"Deploying {client}")
    await ue.deploy(components=[client], quiet=True)


In [ ]:
from datetime import UTC, datetime
from orchestrator_monitoring.artifacts import ExperimentRun
from orchestrator.utils.ansible import get_ansible_inventory_line, run_ansible_command, run_ansible_playbook

SERVER_NODE = "node1"
DURATION, BASE_PORT = 20, 5202
BASELINE_SECONDS, COLLECTION_SECONDS = 5, 5
UE_INTERFACE = "oaitun_ue1"

server_node = ue.deployment_model.nodes[SERVER_NODE]
server_ip = server_node.ssh_info.hostname
server_inventory = get_ansible_inventory_line(SERVER_NODE, server_node)
server_runner = str(Path(ue.deployment_model.params.ansible.runner_path) / "iperf-server")


await run_ansible_playbook(
    playbook=[
        {
            "name": "Install iperf3 on the server VM",
            "hosts": SERVER_NODE,
            "gather_facts": False,
            "become": True,
            "tasks": [
                {
                    "name": "Install iperf3",
                    "ansible.builtin.apt": {
                        "name": "iperf3",
                        "state": "present",
                        "update_cache": True,
                        "cache_valid_time": 3600,
                    },
                }
            ],
        }
    ],
    inventory=server_inventory,
    runner_path=server_runner,
    quiet=True,
)
addresses = await ue.run_command(
    command=["ip", "-j", "-4", "addr", "show", "dev", UE_INTERFACE], components=CLIENTS, quiet=True, flatten=False
)

ue_ips = {}
for name in CLIENTS:
    ipv4 = [a["local"] for device in json.loads(addresses[name]) for a in device["addr_info"] if a["family"] == "inet"]
    if len(ipv4) != 1:
        raise RuntimeError(f"{name}: expected one IPv4 address on {UE_INTERFACE}")
    ue_ips[name] = ipv4[0]

if len(set(ue_ips.values())) != len(CLIENTS):
    raise ValueError("Select UEs with different tunnel addresses")

if not QUERIES_FILE.is_file():
    raise FileNotFoundError(QUERIES_FILE)

print(f"iperf server: {SERVER_NODE} ({server_ip}); UE clients: {ue_ips}")


In [ ]:
async def start_iperf_server(port):
    await run_ansible_command(
        command=["iperf3", "-s", "-D", "-1", "--idle-timeout", "60", "-B", server_ip, "-p", str(port), "-J"],
        hosts=SERVER_NODE,
        inventory=server_inventory,
        runner_path=server_runner,
        quiet=True,
    )


async def iperf_client(client, direction, duration, port):
    reverse = {"uplink": [], "downlink": ["-R"]}[direction]
    output = await ue.run_command(
        command=[
            "iperf3",
            "-c",
            server_ip,
            "-B",
            ue_ips[client],
            "--bind-dev",
            UE_INTERFACE,
            "-p",
            str(port),
            "-t",
            str(duration),
            "--connect-timeout",
            "10000",
            "-J",
            "--get-server-output",
            *reverse,
        ],
        components=[client],
        quiet=True,
        flatten=False,
    )
    return json.loads(output[client])


In [ ]:
SCENARIO = [
    {"name": f"{client}_{direction}", "client": client, "direction": direction, "duration": DURATION}
    for client in CLIENTS
    for direction in ("uplink", "downlink")
]


In [ ]:
from orchestrator_monitoring.artifacts import ExperimentRun

output_directory = Path("results") / f"iperf-{datetime.now(UTC):%Y%m%dT%H%M%S%fZ}"
async with ExperimentRun(
    output_directory,
    name=f"throughput-{CORE_IMPLEMENTATION}",
    monitoring=monitoring,
    queries_file=QUERIES_FILE,
) as experiment:
    experiment.save_json("scenario.json", SCENARIO)
    experiment.save_json("slices.json", SLICE_CONFIG)
    experiment.save_json("ue_addresses.json", ue_ips)

    # Allow a scrape before traffic and one after the final transfer.
    with experiment.section("baseline"):
        await asyncio.sleep(BASELINE_SECONDS)
    for port, step in enumerate(SCENARIO, start=BASE_PORT):
        await start_iperf_server(port)
        with experiment.section(step["name"]):
            report = await iperf_client(step["client"], step["direction"], step["duration"], port)
        experiment.save_json(f"{step['name']}.json", report)
        if report.get("error"):
            raise RuntimeError(f"{step['name']}: {report['error']}")
    with experiment.section("collection"):
        await asyncio.sleep(COLLECTION_SECONDS)
print(f"Artifacts: {output_directory.resolve()}")


## Remove this deployment

The runner performs cleanup automatically. When running cells by hand, run this final cell to destroy these three VMs and their workloads.

In [ ]:
# await prov.unprovision()
